In [1]:
import os
import random
import json
import logging
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm
from sklearn.model_selection import KFold

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

PyTorch version: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 3070 Laptop GPU
VRAM: 8.2 GB


In [2]:
class Config:
    # Пути (измените при необходимости)
    DATA_ROOT = Path("/home/shizm/DL_LABs/DL-lab3/dl-lab-3-product-segmentation/train")
    IMAGES_DIR = DATA_ROOT / "images"
    MASKS_DIR = DATA_ROOT / "masks"
    TEST_IMAGES_DIR = Path("/home/shizm/DL_LABs/DL-lab3/dl-lab-3-product-segmentation/test_images")
    SAVE_DIR = Path("./improved_model_v7")
    
    # Модель
    IMG_SIZE = 448
    ENCODER_NAME = "timm-efficientnet-b5"
    ENCODER_WEIGHTS = "imagenet"
    USE_SCSE = True
    
    # Обучение
    BATCH_SIZE = 4
    GRADIENT_ACCUMULATION = 2          # эффективный батч = 8
    NUM_EPOCHS = 40
    LR_DECODER = 1e-3
    LR_ENCODER = 1e-4
    LR_HEAD = 2e-3
    WEIGHT_DECAY = 1e-4
    
    # K‑Fold
    N_FOLDS = 3
    SEED = 42
    NUM_WORKERS = 4
    
    # Инференс и постобработка
    THRESHOLD = 0.5
    USE_TTA = True
    USE_POSTPROCESSING = True
    PATIENCE = 15
    PP_MIN_AREA = 30
    PP_KERNEL_SIZE = 3
    PP_USE_MEDIAN = True
    PP_MEDIAN_SIZE = 5
    
    # Воспроизводимость
    DETERMINISTIC = True
    
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

config = Config()
config.SAVE_DIR.mkdir(parents=True, exist_ok=True)

if config.DETERMINISTIC:
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    torch.backends.cudnn.benchmark = True

logger.info("Configuration v7")
logger.info(f"Device: {config.DEVICE}")
logger.info(f"Image size: {config.IMG_SIZE}")
logger.info(f"Encoder: {config.ENCODER_NAME}")
logger.info(f"Batch size: {config.BATCH_SIZE} (effective: {config.BATCH_SIZE * config.GRADIENT_ACCUMULATION})")
logger.info(f"Folds: {config.N_FOLDS}")

2026-04-10 13:11:16,974 - INFO - Configuration v7
2026-04-10 13:11:16,975 - INFO - Device: cuda
2026-04-10 13:11:16,976 - INFO - Image size: 448
2026-04-10 13:11:16,976 - INFO - Encoder: timm-efficientnet-b5
2026-04-10 13:11:16,976 - INFO - Batch size: 4 (effective: 8)
2026-04-10 13:11:16,976 - INFO - Folds: 3


In [3]:
def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def find_image_path(images_dir: Path, stem: str) -> Path | None:
    for ext in [".jpg", ".jpeg", ".png"]:
        p = images_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def get_image_mask_pairs(images_dir: Path, masks_dir: Path):
    pairs = []
    for mask_path in masks_dir.glob("*.png"):
        stem = mask_path.stem
        img_path = find_image_path(images_dir, stem)
        if img_path:
            pairs.append((img_path, mask_path))
    return pairs

def get_train_augmentations(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.RandomRotate90(p=0.3),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
        ], p=0.5),
        A.OneOf([
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
            A.GaussianBlur(blur_limit=(3, 5), p=0.3),
        ], p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_val_augmentations(img_size=384):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

class SegmentationDataset(Dataset):
    def __init__(self, image_paths, mask_paths, augmentations=None):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.augmentations = augmentations
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img = cv2.imread(str(self.image_paths[idx]))
        if img is None:
            raise FileNotFoundError(f"Image not found: {self.image_paths[idx]}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(str(self.mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(f"Mask not found: {self.mask_paths[idx]}")
        mask = (mask > 0).astype(np.float32)
        
        if self.augmentations:
            augmented = self.augmentations(image=img, mask=mask)
            img = augmented['image']
            mask = augmented['mask']
        else:
            img = img / 255.0
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).unsqueeze(0).float()
        
        return img, mask

logger.info("Data utilities ready")

2026-04-10 13:11:16,987 - INFO - Data utilities ready


In [4]:
class ImprovedUNet(nn.Module):
    def __init__(self, encoder_name="timm-efficientnet-b5", encoder_weights="imagenet", use_scse=True):
        super().__init__()
        decoder_attention = 'scse' if use_scse else None
        self.backbone = smp.UnetPlusPlus(
            encoder_name=encoder_name,
            encoder_weights=encoder_weights,
            in_channels=3,
            classes=1,
            activation=None,
            decoder_attention_type=decoder_attention
        )
    def forward(self, x):
        return self.backbone(x)

def build_model():
    model = ImprovedUNet(
        encoder_name=config.ENCODER_NAME,
        encoder_weights=config.ENCODER_WEIGHTS,
        use_scse=config.USE_SCSE
    )
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    logger.info(f"Model: {config.ENCODER_NAME}")
    logger.info(f"Total parameters: {total_params:,}")
    logger.info(f"Trainable parameters: {trainable_params:,}")
    return model

class ComboLossV7(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = smp.losses.DiceLoss(mode='binary', from_logits=True)
        self.focal = smp.losses.FocalLoss(mode='binary', alpha=0.75, gamma=2.0)
        self.bce = nn.BCEWithLogitsLoss()
        self.lovasz = smp.losses.LovaszLoss(mode='binary', from_logits=True)
        self.tversky = smp.losses.TverskyLoss(mode='binary', from_logits=True, alpha=0.7, beta=0.3)
    
    def forward(self, pred, target):
        pred = torch.clamp(pred, min=-10, max=10)
        dice = self.dice(pred, target)
        focal = self.focal(pred, target)
        bce = self.bce(pred, target)
        lovasz = self.lovasz(pred, target)
        tversky = self.tversky(pred, target)
        total = dice + 0.5*focal + 0.2*bce + 0.3*lovasz + 0.3*tversky
        if torch.isnan(total) or torch.isinf(total):
            return torch.tensor(1.0, device=pred.device, requires_grad=True)
        return total

logger.info("Model and loss ready")

2026-04-10 13:11:16,997 - INFO - Model and loss ready


In [5]:
def calculate_dice(pred, target, threshold=0.5, eps=1e-7):
    pred = (torch.sigmoid(pred) > threshold).float()
    pred = pred.view(pred.size(0), -1)
    target = target.view(target.size(0), -1)
    intersection = (pred * target).sum(dim=1)
    dice = (2. * intersection + eps) / (pred.sum(dim=1) + target.sum(dim=1) + eps)
    return dice.mean().item()

def calculate_iou(pred, target, threshold=0.5, eps=1e-7):
    pred = (torch.sigmoid(pred) > threshold).float()
    pred = pred.view(pred.size(0), -1)
    target = target.view(target.size(0), -1)
    intersection = (pred * target).sum(dim=1)
    union = pred.sum(dim=1) + target.sum(dim=1) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

def train_one_epoch(model, loader, optimizer, criterion, device, scaler, accum_steps):
    model.train()
    epoch_loss = 0
    epoch_dice = 0
    epoch_iou = 0
    optimizer.zero_grad()
    valid_batches = 0
    
    pbar = tqdm(loader, desc="Training")
    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        
        with autocast():
            outputs = model(images)
            masks = masks.squeeze(1)
            outputs = outputs.squeeze(1)
            loss = criterion(outputs, masks)
        
        if torch.isnan(loss) or torch.isinf(loss):
            optimizer.zero_grad()
            continue
        
        loss = loss / accum_steps
        scaler.scale(loss).backward()
        
        if (batch_idx + 1) % accum_steps == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        epoch_loss += loss.item() * accum_steps
        epoch_dice += calculate_dice(outputs, masks)
        epoch_iou += calculate_iou(outputs, masks)
        valid_batches += 1
        
        pbar.set_postfix({
            'loss': loss.item() * accum_steps,
            'dice': epoch_dice / valid_batches,
            'iou': epoch_iou / valid_batches
        })
    
    n = max(valid_batches, 1)
    return epoch_loss / n, epoch_dice / n, epoch_iou / n

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    epoch_dice = 0
    epoch_iou = 0
    
    for images, masks in tqdm(loader, desc="Validation"):
        images = images.to(device)
        masks = masks.to(device)
        outputs = model(images)
        masks = masks.squeeze(1)
        outputs = outputs.squeeze(1)
        loss = criterion(outputs, masks)
        
        epoch_loss += loss.item()
        epoch_dice += calculate_dice(outputs, masks)
        epoch_iou += calculate_iou(outputs, masks)
    
    n = len(loader)
    return epoch_loss / n, epoch_dice / n, epoch_iou / n

logger.info("Training functions ready")

2026-04-10 13:11:17,008 - INFO - Training functions ready


In [6]:
def run_kfold_training():
    seed_everything(config.SEED)
    pairs = get_image_mask_pairs(config.IMAGES_DIR, config.MASKS_DIR)
    logger.info(f"Total samples: {len(pairs)}")
    
    kf = KFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)
    fold_scores = []
    all_histories = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(pairs)):
        logger.info(f"\n{'='*60}\nFold {fold+1}/{config.N_FOLDS}\nTrain: {len(train_idx)}, Val: {len(val_idx)}\n{'='*60}")
        
        train_images = [pairs[i][0] for i in train_idx]
        train_masks = [pairs[i][1] for i in train_idx]
        val_images = [pairs[i][0] for i in val_idx]
        val_masks = [pairs[i][1] for i in val_idx]
        
        train_ds = SegmentationDataset(train_images, train_masks, get_train_augmentations(config.IMG_SIZE))
        val_ds = SegmentationDataset(val_images, val_masks, get_val_augmentations(config.IMG_SIZE))
        
        train_loader = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True,
                                  num_workers=config.NUM_WORKERS, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=config.BATCH_SIZE, shuffle=False,
                                num_workers=config.NUM_WORKERS, pin_memory=True)
        
        model = build_model().to(config.DEVICE)
        optimizer = torch.optim.AdamW([
            {'params': model.backbone.encoder.parameters(), 'lr': config.LR_ENCODER, 'weight_decay': config.WEIGHT_DECAY},
            {'params': model.backbone.decoder.parameters(), 'lr': config.LR_DECODER, 'weight_decay': config.WEIGHT_DECAY},
            {'params': model.backbone.segmentation_head.parameters(), 'lr': config.LR_HEAD, 'weight_decay': config.WEIGHT_DECAY}
        ])
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-7)
        criterion = ComboLossV7()
        scaler = GradScaler()
        
        best_dice = 0
        patience = 0
        history = []
        
        for epoch in range(1, config.NUM_EPOCHS + 1):
            logger.info(f"\nEpoch {epoch}/{config.NUM_EPOCHS}")
            logger.info("-" * 40)
            train_loss, train_dice, train_iou = train_one_epoch(
                model, train_loader, optimizer, criterion, config.DEVICE, scaler, config.GRADIENT_ACCUMULATION
            )
            val_loss, val_dice, val_iou = validate(model, val_loader, criterion, config.DEVICE)
            scheduler.step()
            current_lr = optimizer.param_groups[1]['lr']
            
            history.append({
                'epoch': epoch, 'fold': fold,
                'train_loss': train_loss, 'train_dice': train_dice, 'train_iou': train_iou,
                'val_loss': val_loss, 'val_dice': val_dice, 'val_iou': val_iou, 'lr': current_lr
            })
            
            logger.info(f"Train - Loss: {train_loss:.4f}, Dice: {train_dice:.4f}, IoU: {train_iou:.4f}")
            logger.info(f"Val   - Loss: {val_loss:.4f}, Dice: {val_dice:.4f}, IoU: {val_iou:.4f}")
            logger.info(f"LR: {current_lr:.6f}")
            
            if val_dice > best_dice:
                best_dice = val_dice
                patience = 0
                torch.save({
                    'fold': fold, 'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'val_dice': val_dice, 'val_iou': val_iou,
                    'config': {
                        'encoder_name': config.ENCODER_NAME,
                        'img_size': config.IMG_SIZE,
                        'use_scse': config.USE_SCSE,
                    },
                    'history': history
                }, config.SAVE_DIR / f'best_model_fold_{fold}.pth')
                logger.info(f"✓ New best model for fold {fold}! Dice: {val_dice:.4f}")
            else:
                patience += 1
                logger.info(f"Best Dice so far: {best_dice:.4f}")
            
            if patience >= config.PATIENCE:
                logger.info(f"Early stopping at epoch {epoch}")
                break
            
            if epoch % 10 == 0:
                torch.save({
                    'fold': fold, 'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'val_dice': val_dice,
                    'config': {'encoder_name': config.ENCODER_NAME, 'img_size': config.IMG_SIZE},
                }, config.SAVE_DIR / f'checkpoint_fold_{fold}_epoch_{epoch}.pth')
        
        fold_scores.append(best_dice)
        all_histories.append(history)
        del model
        torch.cuda.empty_cache()
    
    logger.info("\n" + "="*60)
    logger.info("K-FOLD RESULTS")
    logger.info(f"Scores: {[f'{s:.4f}' for s in fold_scores]}")
    logger.info(f"Mean Dice: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")
    logger.info("="*60)
    
    pd.DataFrame(sum(all_histories, [])).to_csv(config.SAVE_DIR / 'kfold_history.csv', index=False)
    return fold_scores, all_histories

fold_scores, histories = run_kfold_training()

2026-04-10 13:11:17,057 - INFO - Total samples: 2000
2026-04-10 13:11:17,059 - INFO - 
Fold 1/3
Train: 1333, Val: 667
2026-04-10 13:11:17,442 - INFO - Model: timm-efficientnet-b5
2026-04-10 13:11:17,443 - INFO - Total parameters: 32,040,993
2026-04-10 13:11:17,443 - INFO - Trainable parameters: 32,040,993
/tmp/ipykernel_68630/3750307718.py:34: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
2026-04-10 13:11:17,619 - INFO - 
Epoch 1/40
2026-04-10 13:11:17,620 - INFO - ----------------------------------------
Training:   0%|          | 0/333 [00:00<?, ?it/s]/tmp/ipykernel_68630/3314626296.py:31: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Validation: 100%|██████████| 167/167 [00:21<00:00,  7.60it/s]
2026-04-10 13:13:19,075 - INFO - Train - Loss: 0.7659, Dice: 0.7042, IoU: 0.5931
2026-04-10 

KeyboardInterrupt: 

In [7]:
class WeightedTTAWrapper:
    def __init__(self, model, img_size=384, device='cuda', preprocess_fn=None):
        self.model = model
        self.img_size = img_size
        self.device = device
        self.preprocess_fn = preprocess_fn
        self.model.eval()
        self.transforms = [
            ('original', lambda x: x, 0.3),
            ('hflip', lambda x: np.fliplr(x).copy(), 0.2),
            ('vflip', lambda x: np.flipud(x).copy(), 0.2),
            ('rot90', lambda x: np.rot90(x, 1).copy(), 0.1),
            ('rot180', lambda x: np.rot90(x, 2).copy(), 0.1),
            ('rot270', lambda x: np.rot90(x, 3).copy(), 0.1),
        ]
        self.inv = {
            'original': lambda x: x,
            'hflip': lambda x: np.fliplr(x).copy(),
            'vflip': lambda x: np.flipud(x).copy(),
            'rot90': lambda x: np.rot90(x, -1).copy(),
            'rot180': lambda x: np.rot90(x, -2).copy(),
            'rot270': lambda x: np.rot90(x, -3).copy(),
        }
    
    @torch.no_grad()
    def predict(self, image):
        # Один ресайз для всех трансформаций
        resized = cv2.resize(image, (self.img_size, self.img_size)).astype(np.float32)
        if self.preprocess_fn:
            resized = self.preprocess_fn(resized)
        else:
            resized = resized / 255.0
        
        orig_h, orig_w = image.shape[:2]
        probs, weights = [], []
        for name, transform, w in self.transforms:
            aug = transform(resized)
            inp = torch.from_numpy(aug.transpose(2,0,1)).float().unsqueeze(0).to(self.device)
            logits = self.model(inp)
            prob = torch.sigmoid(logits)[0,0].cpu().numpy()
            prob = self.inv[name](prob)
            prob = cv2.resize(prob, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
            probs.append(prob)
            weights.append(w)
        weights = np.array(weights).reshape(-1,1,1)
        return np.sum(np.array(probs) * weights, axis=0) / np.sum(weights)

logger.info("Weighted TTA wrapper ready")

2026-04-10 15:08:50,220 - INFO - Weighted TTA wrapper ready


In [8]:
def post_process_mask(mask, min_area=config.PP_MIN_AREA, kernel_size=config.PP_KERNEL_SIZE,
                      use_median=config.PP_USE_MEDIAN, median_size=config.PP_MEDIAN_SIZE):
    mask = (mask * 255).astype(np.uint8)
    if use_median:
        mask = cv2.medianBlur(mask, median_size)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    if min_area > 0:
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        clean = np.zeros_like(mask)
        for cnt in contours:
            if cv2.contourArea(cnt) >= min_area:
                cv2.drawContours(clean, [cnt], -1, 255, -1)
        mask = clean
    return (mask > 0).astype(np.uint8)

logger.info("Post-processing ready")

2026-04-10 15:08:51,220 - INFO - Post-processing ready


In [9]:
from segmentation_models_pytorch.encoders import get_preprocessing_fn

def ensemble_inference():
    import glob
    model_paths = sorted(glob.glob(str(config.SAVE_DIR / "best_model_fold_*.pth")))
    if not model_paths:
        raise RuntimeError("No trained models found")
    
    models_info = []
    for path in model_paths:
        ckpt = torch.load(path, map_location='cpu')
        cfg = ckpt['config']
        model = ImprovedUNet(
            encoder_name=cfg['encoder_name'],
            encoder_weights=None,
            use_scse=cfg.get('use_scse', True)
        ).to(config.DEVICE)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        
        preprocess_fn = None
        if config.ENCODER_WEIGHTS:
            try:
                preprocess_fn = get_preprocessing_fn(cfg['encoder_name'], pretrained='imagenet')
            except Exception:
                # fallback нормализация ImageNet
                preprocess_fn = lambda x: (x - np.array([0.485,0.456,0.406])) / np.array([0.229,0.224,0.225])
        
        models_info.append({
            'model': model,
            'img_size': cfg['img_size'],
            'preprocess_fn': preprocess_fn,
            'val_dice': ckpt['val_dice'],
        })
        logger.info(f"Loaded {Path(path).name} (Dice: {ckpt['val_dice']:.4f})")
    
    total = sum(m['val_dice'] for m in models_info)
    for m in models_info:
        m['weight'] = m['val_dice'] / total
    
    # Создаём TTA обёртки один раз
    tta_wrappers = [(WeightedTTAWrapper(m['model'], m['img_size'], config.DEVICE, m['preprocess_fn']), m['weight'])
                    for m in models_info]
    
    test_images = sorted([p for p in config.TEST_IMAGES_DIR.glob("*") if p.suffix.lower() in ['.jpg','.jpeg','.png']])
    logger.info(f"Found {len(test_images)} test images")
    
    results = []
    for img_path in tqdm(test_images, desc="Ensemble Inference"):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        avg_prob = None
        total_weight = 0.0
        for tta, w in tta_wrappers:
            prob = tta.predict(img)
            if avg_prob is None:
                avg_prob = prob * w
            else:
                avg_prob += prob * w
            total_weight += w
        avg_prob /= total_weight
        
        mask = (avg_prob > config.THRESHOLD).astype(np.uint8)
        if config.USE_POSTPROCESSING:
            mask = post_process_mask(mask)
        
        results.append({
            'ImageId': img_path.name,
            'mask': json.dumps(mask.tolist(), separators=(',', ':'))
        })
    
    submission_df = pd.DataFrame(results)
    submission_df.to_csv('submission_v7.csv', index=False)
    logger.info(f"Saved submission_v7.csv, {len(results)} rows")
    return submission_df

submission = ensemble_inference()

/tmp/ipykernel_68630/1433551385.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location='cpu')
2026-04-10 15:08:52,816 - INFO - Loaded best

In [10]:
def validate_submission(file='submission_v7.csv'):
    df = pd.read_csv(file)
    print(f"Rows: {len(df)}")
    print(df.head(3))
    sample_mask = json.loads(df.iloc[0]['mask'])
    print(f"Mask shape: {np.array(sample_mask).shape}")
    print(f"Unique values: {np.unique(sample_mask)}")
    assert df['ImageId'].dtype in [object, 'string']
    assert df['mask'].dtype in [object, 'string']
    print("✓ Submission format OK")

validate_submission('submission_v7.csv')

Rows: 2000
                                             ImageId  \
0  10.107.215.111_20260119142748_a0be018c-d498-4b...   
1  10.107.215.111_20260119203335_58a6ef5c-c360-4e...   
2  10.107.215.111_20260119203341_bb74ac11-3f14-4a...   

                                                mask  
0  [[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...  
1  [[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...  
2  [[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...  
Mask shape: (320, 320)
Unique values: [0 1]
✓ Submission format OK
